# Fase C — SFT de la inducción sobre data compositiva (AGI-2)

Continúa el SFT del BARC-8B con la **data compositiva de 2 pasos** generada en la Fase B
(13.271 tareas verificadas), **mezclada** con data original de BARC para no perder la habilidad
base (como el `dataset_mixer` de la receta de BARC). Infra validada en la Fase A:
transformers+peft, `device_map=auto` (8B repartido en 4 L4s), `DevTrainer` device-aware.
Salida: adapter LoRA → lo carga el probe para medir si el verifiable-hit rate sube del 6.25%.

In [ ]:
import os, glob, json, time, random
os.environ['HF_HUB_OFFLINE'] = '1'; os.environ['TRANSFORMERS_OFFLINE'] = '1'
mdir = None
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'config.json' in files and any(f.endswith(('.safetensors', '.bin')) for f in files):
        mdir = root; break
assert mdir, 'no se encontro el modelo BARC'
comp = glob.glob('/kaggle/input/**/compositional_tasks.jsonl', recursive=True)
orig = glob.glob('/kaggle/input/**/induction_heavy_slice.jsonl', recursive=True)
assert comp, 'no se encontro la data compositiva (Fase B)'
print('modelo:', mdir)
print('compositiva:', comp[0])
print('original:', orig[0] if orig else '(no adjunta)')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
MAXLEN = 2048
tok = AutoTokenizer.from_pretrained(mdir, local_files_only=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(mdir, torch_dtype=torch.bfloat16,
                                             device_map='auto', local_files_only=True,
                                             attn_implementation='sdpa')
model.config.use_cache = False
lcfg = LoraConfig(r=32, lora_alpha=32, lora_dropout=0.0, bias='none', task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
model = get_peft_model(model, lcfg)
model.print_trainable_parameters()

In [ ]:
from datasets import load_dataset, concatenate_datasets
# MEZCLA (estilo dataset_mixer de BARC): compositiva (nueva, AGI-2) + original (base, anti-olvido)
ds_c = load_dataset('json', data_files=comp[0], split='train')
parts = [ds_c]
if orig:
    ds_o = load_dataset('json', data_files=orig[0], split='train')
    n_o = min(len(ds_o), len(ds_c))        # mezcla 1:1 (v2: mas anclaje de estilo)
    parts.append(ds_o.shuffle(seed=42).select(range(n_o)))
    print(f'mezcla: {len(ds_c)} compositivas + {n_o} originales')
raw = concatenate_datasets([p.select_columns(['messages']) for p in parts]).shuffle(seed=42)
print('total ejemplos:', len(raw))

def tokenize(ex):
    msgs = ex['messages']
    full = tok.apply_chat_template(msgs, tokenize=False)
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
    fids = tok(full, truncation=True, max_length=MAXLEN, add_special_tokens=False)['input_ids']
    pids = tok(prompt, truncation=True, max_length=MAXLEN, add_special_tokens=False)['input_ids']
    n_p = min(len(pids), len(fids))
    return {'input_ids': fids, 'attention_mask': [1]*len(fids), 'labels': [-100]*n_p + fids[n_p:]}

ds = raw.map(tokenize, remove_columns=raw.column_names, num_proc=4)
ds = ds.filter(lambda e: len(e['input_ids']) <= MAXLEN and any(l != -100 for l in e['labels']))
print('tras tokenizar/filtrar:', len(ds))

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
import torch

class DevTrainer(Trainer):
    """compute_loss que mueve las labels al device de los logits (modelo repartido)."""
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop('labels')
        out = model(**inputs)
        logits = out.logits.float()
        labels = labels.to(logits.device)
        sl = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))
        st = labels[..., 1:].contiguous().view(-1)
        loss = torch.nn.functional.cross_entropy(sl, st, ignore_index=-100)
        return (loss, out) if return_outputs else loss

MAX_STEPS = int(os.getenv('MAX_STEPS', '900'))   # ~7200 ejemplos vistos (batch efectivo 8)
collator = DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100)
args = TrainingArguments(
    output_dir='/kaggle/working/sft_out', per_device_train_batch_size=1,
    gradient_accumulation_steps=8, warmup_steps=20, max_steps=MAX_STEPS, learning_rate=1e-4,
    logging_steps=10, bf16=True, optim='adamw_torch', lr_scheduler_type='cosine',
    seed=42, report_to='none', save_strategy='no')
trainer = DevTrainer(model=model, args=args, train_dataset=ds, data_collator=collator)
t0 = time.time()
stats = trainer.train()
print(f'SFT OK en {(time.time()-t0)/60:.1f} min | train_loss: {stats.training_loss:.4f}')

In [ ]:
model.save_pretrained('/kaggle/working/barc_comp_adapter')
tok.save_pretrained('/kaggle/working/barc_comp_adapter')
losses = [h['loss'] for h in trainer.state.log_history if 'loss' in h]
res = {'n_ejemplos': len(ds), 'max_steps': MAX_STEPS,
       'loss_inicial': losses[0] if losses else None,
       'loss_final': losses[-1] if losses else None,
       'loss_curva': [round(x, 4) for x in losses[::max(1, len(losses)//12)]]}
json.dump(res, open('/kaggle/working/sft_c_result.json', 'w'), indent=1)
print('SFT FASE C:', json.dumps(res)[:400])
print('adapter -> /kaggle/working/barc_comp_adapter')